In [1]:
from collections import defaultdict
from itertools import combinations
import math

In [2]:
transactions = [
    ["Milk", "Bread", "Butter"],
    ["Bread", "Butter"],
    ["Milk", "Bread"],
    ["Milk", "Butter"],
    ["Bread", "Butter", "Eggs"],
    ["Milk", "Bread", "Butter"],
    ["Bread", "Eggs"],
    ["Milk", "Bread", "Eggs"],
    ["Milk", "Butter"],
    ["Bread", "Butter", "Eggs"]
]

print("Number of Transactions:", len(transactions))

for i, transaction in enumerate(transactions, start=1):
    print(i, transaction)

Number of Transactions: 10
1 ['Milk', 'Bread', 'Butter']
2 ['Bread', 'Butter']
3 ['Milk', 'Bread']
4 ['Milk', 'Butter']
5 ['Bread', 'Butter', 'Eggs']
6 ['Milk', 'Bread', 'Butter']
7 ['Bread', 'Eggs']
8 ['Milk', 'Bread', 'Eggs']
9 ['Milk', 'Butter']
10 ['Bread', 'Butter', 'Eggs']


In [3]:
min_support = 0.3

min_support_count = math.ceil(
    len(transactions) * min_support
)

print("Minimum Support:", min_support)
print("Minimum Support Count:", min_support_count)

Minimum Support: 0.3
Minimum Support Count: 3


In [4]:
def partition_transactions(transactions, number_of_partitions):
    partitions = [[] for _ in range(number_of_partitions)]

    for index, transaction in enumerate(transactions):
        partition_index = index % number_of_partitions
        partitions[partition_index].append(set(transaction))

    return partitions

In [5]:
number_of_partitions = 2

partitions = partition_transactions(
    transactions,
    number_of_partitions
)

for i, partition in enumerate(partitions, start=1):
    print(f"Partition {i}:")
    for transaction in partition:
        print(transaction)
    print()

Partition 1:
{'Bread', 'Milk', 'Butter'}
{'Bread', 'Milk'}
{'Bread', 'Eggs', 'Butter'}
{'Bread', 'Eggs'}
{'Milk', 'Butter'}

Partition 2:
{'Bread', 'Butter'}
{'Milk', 'Butter'}
{'Bread', 'Milk', 'Butter'}
{'Bread', 'Eggs', 'Milk'}
{'Bread', 'Eggs', 'Butter'}



In [6]:
def count_single_items(partition):
    counts = defaultdict(int)

    for transaction in partition:
        for item in transaction:
            counts[frozenset([item])] += 1

    return counts

In [7]:
def combine_counts(partition_counts):
    total_counts = defaultdict(int)

    for counts in partition_counts:
        for itemset, count in counts.items():
            total_counts[itemset] += count

    return total_counts

In [8]:
def get_frequent_itemsets(item_counts, min_support_count):
    frequent = {}

    for itemset, count in item_counts.items():
        if count >= min_support_count:
            frequent[itemset] = count

    return frequent

In [9]:
partition_counts = []

for partition in partitions:
    counts = count_single_items(partition)
    partition_counts.append(counts)

item_counts = combine_counts(partition_counts)

frequent_1 = get_frequent_itemsets(
    item_counts,
    min_support_count
)

print("Frequent 1-Itemsets:")

for itemset, count in frequent_1.items():
    print(set(itemset), "->", count)

Frequent 1-Itemsets:
{'Bread'} -> 8
{'Milk'} -> 6
{'Butter'} -> 7
{'Eggs'} -> 4


In [10]:
def generate_candidates(previous_frequent, k):
    candidates = set()

    previous_itemsets = list(previous_frequent.keys())

    for i in range(len(previous_itemsets)):
        for j in range(i + 1, len(previous_itemsets)):

            union_set = previous_itemsets[i] | previous_itemsets[j]

            if len(union_set) == k:
                candidates.add(frozenset(union_set))

    return candidates

In [11]:
class HashTreeNode:
    def __init__(self):
        self.children = {}
        self.candidates = []

In [12]:
class HashTree:
    def __init__(self):
        self.root = HashTreeNode()

    def insert(self, itemset):
        current = self.root

        for item in sorted(itemset):
            if item not in current.children:
                current.children[item] = HashTreeNode()

            current = current.children[item]

        current.candidates.append(frozenset(itemset))

In [13]:
def verify_candidate(tree, itemset):
    current = tree.root

    for item in sorted(itemset):
        if item not in current.children:
            return False

        current = current.children[item]

    return frozenset(itemset) in current.candidates

In [14]:
def build_hash_tree(candidates):
    tree = HashTree()

    for candidate in candidates:
        tree.insert(candidate)

    return tree

In [15]:
def count_candidates_in_partition(
    partition,
    candidates,
    hash_tree
):
    counts = defaultdict(int)

    for transaction in partition:

        for candidate in candidates:

            if not verify_candidate(hash_tree, candidate):
                continue

            if candidate.issubset(transaction):
                counts[candidate] += 1

    return counts

In [16]:
def apriori(
    transactions,
    min_support,
    number_of_partitions
):

    min_support_count = math.ceil(
        len(transactions) * min_support
    )

    partitions = partition_transactions(
        transactions,
        number_of_partitions
    )

    # -------------------------
    # 1-Itemsets
    # -------------------------

    partition_counts = []

    for partition in partitions:
        counts = count_single_items(partition)
        partition_counts.append(counts)

    total_counts = combine_counts(partition_counts)

    frequent = get_frequent_itemsets(
        total_counts,
        min_support_count
    )

    all_frequent = dict(frequent)

    k = 2

    # -------------------------
    # Larger Itemsets
    # -------------------------

    while frequent:

        candidates = generate_candidates(
            frequent,
            k
        )

        if not candidates:
            break

        # Build Hash Tree
        hash_tree = build_hash_tree(candidates)

        # Count candidates in partitions
        partition_candidate_counts = []

        for partition in partitions:

            counts = count_candidates_in_partition(
                partition,
                candidates,
                hash_tree
            )

            partition_candidate_counts.append(counts)

        total_candidate_counts = combine_counts(
            partition_candidate_counts
        )

        frequent = get_frequent_itemsets(
            total_candidate_counts,
            min_support_count
        )

        for itemset, count in frequent.items():
            all_frequent[itemset] = count

        k += 1

    return all_frequent

In [17]:
frequent_itemsets = apriori(
    transactions,
    min_support,
    number_of_partitions
)

print("Frequent Itemsets")
print("-----------------")

for itemset, count in sorted(
    frequent_itemsets.items(),
    key=lambda x: (len(x[0]), sorted(x[0]))
):
    print(set(itemset), "->", count)

Frequent Itemsets
-----------------
{'Bread'} -> 8
{'Butter'} -> 7
{'Eggs'} -> 4
{'Milk'} -> 6
{'Bread', 'Butter'} -> 5
{'Bread', 'Eggs'} -> 4
{'Bread', 'Milk'} -> 4
{'Milk', 'Butter'} -> 4


In [18]:
def calculate_support(itemset, transactions):
    count = 0

    itemset = set(itemset)

    for transaction in transactions:
        if itemset.issubset(set(transaction)):
            count += 1

    return count / len(transactions)

In [19]:
def generate_rules(
    frequent_itemsets,
    transactions,
    min_confidence=0.5
):

    rules = []

    for itemset in frequent_itemsets:

        if len(itemset) < 2:
            continue

        itemset = set(itemset)

        for size in range(1, len(itemset)):

            for antecedent_tuple in combinations(
                itemset,
                size
            ):

                antecedent = set(antecedent_tuple)
                consequent = itemset - antecedent

                support_itemset = calculate_support(
                    itemset,
                    transactions
                )

                support_antecedent = calculate_support(
                    antecedent,
                    transactions
                )

                if support_antecedent == 0:
                    continue

                confidence = (
                    support_itemset /
                    support_antecedent
                )

                if confidence >= min_confidence:

                    rules.append({
                        "antecedent": antecedent,
                        "consequent": consequent,
                        "support": support_itemset,
                        "confidence": confidence
                    })

    return rules

In [20]:
def calculate_lift(
    support_itemset,
    support_antecedent,
    support_consequent
):

    if support_antecedent == 0 or support_consequent == 0:
        return 0

    return (
        support_itemset /
        (support_antecedent * support_consequent)
    )

In [21]:
def calculate_conviction(
    confidence,
    support_consequent
):

    if confidence >= 1:
        return float("inf")

    if support_consequent == 1:
        return float("inf")

    return (
        (1 - support_consequent) /
        (1 - confidence)
    )

In [22]:
rules = generate_rules(
    frequent_itemsets,
    transactions,
    min_confidence=0.5
)

for rule in rules:

    antecedent = rule["antecedent"]
    consequent = rule["consequent"]

    support_itemset = rule["support"]
    confidence = rule["confidence"]

    support_antecedent = calculate_support(
        antecedent,
        transactions
    )

    support_consequent = calculate_support(
        consequent,
        transactions
    )

    lift = calculate_lift(
        support_itemset,
        support_antecedent,
        support_consequent
    )

    conviction = calculate_conviction(
        confidence,
        support_consequent
    )

    rule["lift"] = lift
    rule["conviction"] = conviction

In [23]:
print("Association Rules")
print("==================")

for rule in rules:

    print(
        f"{rule['antecedent']} "
        f"-> "
        f"{rule['consequent']}"
    )

    print(
        f"Support    : {rule['support']:.3f}"
    )

    print(
        f"Confidence : {rule['confidence']:.3f}"
    )

    print(
        f"Lift       : {rule['lift']:.3f}"
    )

    print(
        f"Conviction : {rule['conviction']:.3f}"
    )

    print("------------------")

Association Rules
{'Bread'} -> {'Butter'}
Support    : 0.500
Confidence : 0.625
Lift       : 0.893
Conviction : 0.800
------------------
{'Butter'} -> {'Bread'}
Support    : 0.500
Confidence : 0.714
Lift       : 0.893
Conviction : 0.700
------------------
{'Bread'} -> {'Milk'}
Support    : 0.400
Confidence : 0.500
Lift       : 0.833
Conviction : 0.800
------------------
{'Milk'} -> {'Bread'}
Support    : 0.400
Confidence : 0.667
Lift       : 0.833
Conviction : 0.600
------------------
{'Milk'} -> {'Butter'}
Support    : 0.400
Confidence : 0.667
Lift       : 0.952
Conviction : 0.900
------------------
{'Butter'} -> {'Milk'}
Support    : 0.400
Confidence : 0.571
Lift       : 0.952
Conviction : 0.933
------------------
{'Bread'} -> {'Eggs'}
Support    : 0.400
Confidence : 0.500
Lift       : 1.250
Conviction : 1.200
------------------
{'Eggs'} -> {'Bread'}
Support    : 0.400
Confidence : 1.000
Lift       : 1.250
Conviction : inf
------------------
